# Valutazione migliori checkpoint

Questo notebook lancia lo script che valuta automaticamente il miglior checkpoint disponibile per ogni config di fine-tuning e salva metriche, predizioni e riepilogo aggregato.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Esegui il notebook dal repository SHIELD o da una sua sottocartella.")

print(PROJECT_ROOT)

## Smoke test

Esegui prima questa cella per verificare che caricamento modello, checkpoint e test set funzionino. Valuta solo 3 esempi.

In [ ]:
import subprocess

cmd = [
    "uv", "run", "python", "scripts/evaluation/evaluate_best_checkpoints.py",
    "--all-finetuning-configs",
    "--limit", "3",
    "--output-dir", "outputs/evaluation/best_checkpoints_smoke",
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

## Valutazione completa

Esegui questa cella quando lo smoke test e' andato a buon fine. Salva un file per-run e un riepilogo aggregato in `outputs/evaluation/best_checkpoints/`.

In [ ]:
import subprocess

cmd = [
    "uv", "run", "python", "scripts/evaluation/evaluate_best_checkpoints.py",
    "--all-finetuning-configs",
    "--output-dir", "outputs/evaluation/best_checkpoints",
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

## Leggi riepilogo

Mostra il confronto finale tra i migliori checkpoint valutati.

In [ ]:
import pandas as pd

summary_path = PROJECT_ROOT / "outputs/evaluation/best_checkpoints/summary.csv"
df = pd.read_csv(summary_path)
df.sort_values("bertscore_f1", ascending=False)